# 08 — Table D.1, height-balanced MAE, and height-stratified MAE

This notebook rebuilds Table D.1 from the authoritative class-wise results, recomputes conventional MAE and height-balanced MAE (aMAE), and reproduces the targeted Agadir upper-class sensitivity calculation.

In [ ]:
from pathlib import Path
import hashlib
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display

NATURAL_ROOT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
INPUT_CSV = (
    NATURAL_ROOT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1"
    / "CHM_Comparison" / "four_product_common_support_height_class_mae.csv"
)
MANUSCRIPT_TEX = (
    NATURAL_ROOT / "Writing_Article" / "1_Article" / "Final_Version_3_aMAE"
    / "Reflexion" / "supplementary" / "supp_04_external_product_provenance.tex"
)
OUTPUT_DIR = (
    NATURAL_ROOT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1"
    / "CHM_Comparison" / "Table_D1_Recalculation"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRODUCTS = ["Our model", "Pa24", "L23", "T24"]
MAE_COLS = {p: f"{p}_MAE_m" for p in PRODUCTS}
SITE_ORDER = ["Ifran", "Maamoura", "Agadir"]
EXPECTED_SUPPORT = {"Ifran": 5053, "Maamoura": 1796, "Agadir": 6401}

print(f"Input: {INPUT_CSV}", flush=True)
print(f"Current manuscript table: {MANUSCRIPT_TEX}", flush=True)
print(f"Outputs: {OUTPUT_DIR}", flush=True)

In [ ]:
df = pd.read_csv(INPUT_CSV)
required = {"Study_area", "Support", "support_n", "height_class", "class_n", *MAE_COLS.values()}
missing = required.difference(df.columns)
assert not missing, f"Missing columns: {sorted(missing)}"
assert df[list(MAE_COLS.values())].apply(np.isfinite).all().all(), "Non-finite MAE detected."
assert (df["class_n"] > 0).all(), "Empty or negative class count detected."

def lower_edge(label):
    return float(str(label).replace(" m", "").replace("–", "-").split("-")[0])

df["class_order"] = df["height_class"].map(lower_edge)
df = df.sort_values(
    ["Study_area", "class_order"],
    key=lambda s: s.map({v: i for i, v in enumerate(SITE_ORDER)}) if s.name == "Study_area" else s,
).reset_index(drop=True)

support_audit = df.groupby("Study_area", sort=False).agg(
    declared_support=("support_n", "first"),
    summed_class_n=("class_n", "sum"),
    populated_intervals=("height_class", "size"),
)
support_audit["expected_support"] = pd.Series(EXPECTED_SUPPORT)
support_audit["pass"] = (
    support_audit["declared_support"].eq(support_audit["summed_class_n"])
    & support_audit["declared_support"].eq(support_audit["expected_support"])
)
display(support_audit)
assert support_audit["pass"].all(), "Common-support counts do not reconcile."
print("PASS: all class counts reconcile with the exact common GEDI support.", flush=True)

In [ ]:
exact_cols = ["Study_area", "height_class", "class_n", *MAE_COLS.values()]
table_exact = df[exact_cols].copy()
table_rounded = table_exact.copy()
for col in MAE_COLS.values():
    table_rounded[col] = table_rounded[col].round(2)

table_exact.to_csv(OUTPUT_DIR / "table_D1_recalculated_unrounded.csv", index=False)
table_rounded.to_csv(OUTPUT_DIR / "table_D1_recalculated_2dp.csv", index=False)
display(table_rounded)

latex_rows = []
for site in SITE_ORDER:
    part = table_exact[table_exact["Study_area"].eq(site)]
    for row_i, (_, row) in enumerate(part.iterrows()):
        values = {p: float(row[MAE_COLS[p]]) for p in PRODUCTS}
        best = min(values.values())
        rendered = {
            p: (rf"\textbf{{{values[p]:.2f}}}" if np.isclose(values[p], best) else f"{values[p]:.2f}")
            for p in PRODUCTS
        }
        site_cell = rf"\multirow{{{len(part)}}}{{*}}{{{site}}}" if row_i == 0 else ""
        height = str(row["height_class"]).replace(" m", "").replace("-", "--")
        n = f"{int(row['class_n']):,}"
        latex_rows.append(
            f"{site_cell} & {height} & {n} & "
            + " & ".join(rendered[p] for p in PRODUCTS)
            + r" \\" 
        )
    if site != SITE_ORDER[-1]:
        latex_rows.append(r"\midrule")

latex_path = OUTPUT_DIR / "table_D1_recalculated_rows.tex"
latex_path.write_text("\n".join(latex_rows) + "\n", encoding="utf-8")
print(f"Saved publication rows: {latex_path}", flush=True)
print("\n".join(latex_rows), flush=True)

In [ ]:
summary_rows = []
for site in SITE_ORDER:
    part = table_exact[table_exact["Study_area"].eq(site)]
    weights = part["class_n"].to_numpy(float)
    for product in PRODUCTS:
        values = part[MAE_COLS[product]].to_numpy(float)
        summary_rows.append({
            "Study_area": site,
            "Product": product,
            "n": int(weights.sum()),
            "conventional_MAE_m": float(np.average(values, weights=weights)),
            "aMAE_m": float(values.mean()),
            "n_height_intervals": int(len(values)),
        })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / "table_D1_MAE_aMAE_check_unrounded.csv", index=False)
display(summary.assign(
    conventional_MAE_m=summary["conventional_MAE_m"].round(2),
    aMAE_m=summary["aMAE_m"].round(2),
))
print("PASS: MAE and aMAE were reconstructed from unrounded class-wise values.", flush=True)

In [ ]:
agadir_reduced = table_exact[
    table_exact["Study_area"].eq("Agadir")
    & ~table_exact["height_class"].astype(str).str.startswith("15-20")
].copy()

sensitivity = pd.DataFrame({
    "Product": PRODUCTS,
    "mean_classwise_MAE_without_15_20_m": [
        agadir_reduced[MAE_COLS[p]].mean() for p in PRODUCTS
    ],
    "retained_intervals": len(agadir_reduced),
    "excluded_interval": "15-20 m (n=29)",
})
sensitivity.to_csv(OUTPUT_DIR / "agadir_aMAE_sensitivity_without_15_20m.csv", index=False)
display(sensitivity.assign(
    mean_classwise_MAE_without_15_20_m=
        sensitivity["mean_classwise_MAE_without_15_20_m"].round(2)
))

our_value = sensitivity.loc[sensitivity["Product"].eq("Our model"), "mean_classwise_MAE_without_15_20_m"].iat[0]
pa24_value = sensitivity.loc[sensitivity["Product"].eq("Pa24"), "mean_classwise_MAE_without_15_20_m"].iat[0]
print(f"Our model: {our_value:.2f} m; Pa24: {pa24_value:.2f} m", flush=True)
assert pa24_value < our_value, "The reported Agadir ranking reversal did not persist."
print("PASS: Pa24 remains ahead after excluding the uppermost Agadir interval.", flush=True)

In [ ]:
tex = MANUSCRIPT_TEX.read_text(encoding="utf-8")
current_site = None
printed_rows = []

def clean_latex_number(token):
    token = token.strip().replace(r"\textbf{", "").replace("}", "")
    token = token.replace(",", "").replace(r"\\", "").strip()
    return token

for line in tex.splitlines():
    site_match = re.search(r"\\multirow\{\d+\}\{\*\}\{([^}]+)\}", line)
    if site_match:
        current_site = site_match.group(1)
    if current_site and re.search(r"&\s*\d+--\d+\s*&", line):
        parts = [p.strip() for p in line.split("&")]
        height = parts[1].replace("--", "-") + " m"
        n = int(clean_latex_number(parts[2]))
        values = [float(clean_latex_number(parts[k])) for k in range(3, 7)]
        printed_rows.append({
            "Study_area": current_site,
            "height_class": height,
            "class_n_printed": n,
            **{f"{p}_printed": v for p, v in zip(PRODUCTS, values)},
        })

printed = pd.DataFrame(printed_rows)
comparison = table_rounded.merge(printed, on=["Study_area", "height_class"], how="outer", validate="one_to_one")
discrepancies = []
for _, row in comparison.iterrows():
    if int(row["class_n"]) != int(row["class_n_printed"]):
        discrepancies.append({
            "Study_area": row["Study_area"], "height_class": row["height_class"],
            "field": "class_n", "printed": row["class_n_printed"], "correct": row["class_n"],
        })
    for product in PRODUCTS:
        correct = float(row[MAE_COLS[product]])
        printed_value = float(row[f"{product}_printed"])
        if not np.isclose(correct, printed_value, atol=5e-9):
            discrepancies.append({
                "Study_area": row["Study_area"], "height_class": row["height_class"],
                "field": product, "printed": printed_value, "correct": correct,
            })

discrepancies = pd.DataFrame(discrepancies)
discrepancies.to_csv(OUTPUT_DIR / "table_D1_discrepancies_vs_current_manuscript.csv", index=False)
display(discrepancies)
print(f"Detected {len(discrepancies)} manuscript discrepancies.", flush=True)

In [ ]:
manifest = {
    "input_csv": str(INPUT_CSV),
    "input_sha256": hashlib.sha256(INPUT_CSV.read_bytes()).hexdigest(),
    "manuscript_table_source": str(MANUSCRIPT_TEX),
    "rounding_decimals": 2,
    "aMAE_definition": "unweighted mean of unrounded populated height-class MAEs",
    "products": PRODUCTS,
    "sites": SITE_ORDER,
    "outputs": sorted(str(p) for p in OUTPUT_DIR.glob("*")),
}
manifest_path = OUTPUT_DIR / "table_D1_recalculation_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2), flush=True)

## Figure 9

The final cells generate the selected height-stratified MAE figure with class-wise errors and aligned GEDI observation counts. Unused visual alternatives are omitted.

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator

SOURCE_CSV = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling\Results\Final_Article_Harmonized_GEDIAnchored_NaturalP1\CHM_Comparison\four_product_common_support_height_class_mae.csv")
LOCAL_CSV = Path("data") / "four_product_common_support_height_class_mae.csv"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = LOCAL_CSV if LOCAL_CSV.exists() else SOURCE_CSV
if not csv_path.exists():
    raise FileNotFoundError(f"Table 4 source CSV not found: {csv_path}")

df = pd.read_csv(csv_path)
expected = {
    "Study_area", "height_class", "class_n", "Our model_MAE_m",
    "Pa24_MAE_m", "L23_MAE_m", "T24_MAE_m"
}
missing = expected.difference(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {sorted(missing)}")

SITES = ["Ifran", "Maamoura", "Agadir"]
PRODUCTS = ["Our model", "Pa24", "L23", "T24"]
COLS = {p: f"{p}_MAE_m" for p in PRODUCTS}
COLORS = {
    "Our model": "#0072B2", "Pa24": "#E69F00",
    "L23": "#009E73", "T24": "#CC79A7"
}
MARKERS = {"Our model": "o", "Pa24": "s", "L23": "^", "T24": "D"}

def class_lower(label):
    return float(str(label).split("-")[0].replace("–", "-"))

df["class_order"] = df["height_class"].map(class_lower)
df = df.sort_values(["Study_area", "class_order"]).reset_index(drop=True)

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7.5,
    "axes.titlesize": 8.5, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7.5, "figure.dpi": 140,
    "savefig.dpi": 600, "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.spines.top": False, "axes.spines.right": False,
})

def save(fig, stem):
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.png", bbox_inches="tight", dpi=600)
    fig.savefig(OUTPUT_DIR / f"{stem}.tiff", bbox_inches="tight", dpi=600,
                pil_kwargs={"compression": "tiff_lzw"})
    print(f"Saved: {OUTPUT_DIR / (stem + '.pdf')}")

print(f"Loaded {len(df)} class rows from: {csv_path}")
display(df)


In [ ]:
fig = plt.figure(figsize=(9.4, 4.85), constrained_layout=False)
gs = fig.add_gridspec(2, 3, height_ratios=[6.8, 1.15], hspace=0.09, wspace=0.12)
SITE_TITLES = {
    "Ifran": "Ifran - moderately dense",
    "Maamoura": "Maamoura - low-density",
    "Agadir": "Agadir - sparse",
}
WIDTHS = {"Our model": 1.85, "Pa24": 1.85,
          "L23": 2.35, "T24": 1.55}

for j, site in enumerate(SITES):
    d = df[df.Study_area.eq(site)].copy()

    # Plot every class at its true 5-m midpoint. The upper MAE curves and the
    # lower count bars therefore use the same physical x coordinate system.
    labels = d["height_class"].astype(str).str.replace(" m", "", regex=False)
    bounds = np.asarray([
        [float(v) for v in label.replace("–", "-").split("-")]
        for label in labels
    ])
    x = bounds.mean(axis=1)
    bin_widths = bounds[:, 1] - bounds[:, 0]
    bin_edges = np.concatenate((bounds[:, 0], bounds[-1:, 1]))
    x_edges = (float(bounds[:, 0].min()), float(bounds[:, 1].max()))

    ax = fig.add_subplot(gs[0, j])
    axn = fig.add_subplot(gs[1, j], sharex=ax)
    for p in PRODUCTS:
        ax.plot(x, d[COLS[p]], color=COLORS[p], lw=WIDTHS[p],
                label=p, zorder=3)

    ax.set_title(SITE_TITLES[site], fontweight="bold", pad=10, fontsize=10.0)
    ax.text(0.01, 1.02, f"({chr(97+j)})", transform=ax.transAxes,
            ha="left", va="bottom", fontweight="bold", fontsize=9.2)
    ax.set_ylabel("MAE (m)" if j == 0 else "")
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
    site_max = np.nanmax(d[[COLS[p] for p in PRODUCTS]].to_numpy(float))
    ax.set_ylim(0.0, site_max * 1.06)
    ax.set_xlim(*x_edges)
    ax.grid(axis="y", color="0.88", lw=0.6)
    # Curves remain at the class midpoints, whereas the unlabelled tick marks
    # indicate the true 5-m class boundaries (0, 5, 10, ...).
    ax.set_xticks(x)
    ax.set_xticks(bin_edges, minor=True)
    ax.tick_params(axis="x", which="major", bottom=False, labelbottom=False)
    ax.tick_params(axis="x", which="minor", bottom=True, labelbottom=False,
                   length=3.0, width=0.8)

    # Bar heights use a logarithmic scale, but the horizontal geometry uses
    # the same true bin midpoints and 5-m widths as the curves above.
    counts = d["class_n"].to_numpy(float)
    axn.bar(x, counts - 1.0, bottom=1.0, color="0.72",
            width=0.72 * bin_widths, align="center")
    axn.set_xlim(*x_edges)
    axn.set_yscale("log")
    axn.set_ylim(1.0, counts.max() * 3.0)
    axn.set_yticks([])
    axn.tick_params(axis="y", which="both", left=False, labelleft=False)
    axn.spines[["left", "right", "top"]].set_visible(False)
    axn.tick_params(axis="x", which="major", length=2.5, pad=1)
    axn.tick_params(axis="x", which="minor", bottom=False)
    axn.set_xticks(x)
    axn.set_xticklabels(labels, rotation=27, ha="center",
                        rotation_mode="anchor")
    if j == 0:
        axn.set_ylabel("n (log scale)", fontsize=7, rotation=90, labelpad=1)
    for xi, n in zip(x, counts):
        n_label = f"{n/1000:.1f}k" if n >= 1000 else f"{int(n)}"
        axn.annotate(n_label, xy=(xi, n), xytext=(0, 3.0),
                     textcoords="offset points", ha="center", va="bottom",
                     fontsize=7.0, color="0.20", clip_on=False)

handles = [Line2D([0], [0], color=COLORS[p], lw=WIDTHS[p], label=p)
           for p in PRODUCTS]
fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.997), columnspacing=2.1, handletextpad=0.6,
           fontsize=8.8)
fig.supxlabel("GEDI RH95 class (m)", y=0.045, fontsize=8.2)
fig.subplots_adjust(top=0.86, bottom=0.145, left=0.060, right=0.995)
save(fig, "01_recommended_lineplot_with_n")
plt.show()
